# Caderno 2 - Anotação usando LLMs

## 1. Parâmetros

In [1]:
import pandas as pd
import re
import os
from openai import OpenAI
from getpass import getpass
import json

In [2]:
ARQUIVO_QUERY = './dados/outputs/0 - qrel - docs - query - raw_human_eval/query.csv'
ARQUIVO_QREL = './dados/outputs/0 - qrel - docs - query - raw_human_eval/qrel.csv'
ARQUIVO_DOCS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/docs.csv'
ARQUIVO_HUMAN_EVALS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/raw_human_eval.csv'
ARQUIVO_LLM_EVALS = './dados/outputs/2 - llm evals/raw_llm_eval.csv'

In [3]:
OPENAI_KEY = getpass("KEY OpenAI")
DEEPSEEK_KEY = getpass("KEY DeepSeek")
SABIA_KEY = getpass("KEY Maritaca")
GEMINI_KEY = getpass("KEY Gemini")

KEY OpenAI ········
KEY DeepSeek ········
KEY Maritaca ········
KEY Gemini ········


In [4]:
client_openai = OpenAI(api_key=OPENAI_KEY)
client_deepseek = OpenAI(api_key=DEEPSEEK_KEY, base_url="https://api.deepseek.com")
client_sabia = OpenAI(api_key=SABIA_KEY, base_url="https://chat.maritaca.ai/api")
client_gemini = OpenAI(api_key=GEMINI_KEY, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [5]:
CONFIGURACOES_LLM_PARA_TESTAR = [
    {
        "model": "sabiazinho-4-2026-01-06",
        "client": client_sabia
    }
]

In [6]:
MSG_SISTEMA_SIMPLE_PROMPT = """
Você é um avaliador de relevância de normas do Tribunal de Contas da União (TCU).

Sua tarefa é avaliar a relevância de um DOCUMENTO em relação a uma CONSULTA (QUERY), atribuindo um score de relevância em {0, 1, 2}, conforme as regras abaixo.

### REGRAS PARA O SCORE ###:
2 – RELEVANTE: O documento responde diretamente à consulta e contém a informação central solicitada. O conteúdo é suficiente para permitir ao usuário atingir o objetivo da busca. O documento não será considerado relevante para a consulta se apenas mencionar termos semelhantes, mas não tratar efetivamente do objeto da consulta.

1 – POUCO RELEVANTE: O documento menciona o tema da consulta ou trata de assunto relacionado, mas de forma parcial ou insuficiente para atender plenamente à necessidade informacional.

0 – IRRELEVANTE: O documento não trata do assunto da consulta ou aborda tema distinto/incompatível.
### ###

A sua resposta deverá ser apenas um JSON válido, sem qualquer texto antes ou depois, e deve conter apenas o atributo "score".

### REGRAS PARA O JSON ###
"score": O valor do score (0, 1 ou 2) para o grau de relevância estimado.
### ###
""".strip()

In [7]:
MSG_SISTEMA_COT_PROMPT = """
Você é um avaliador de relevância de normas do Tribunal de Contas da União (TCU).

Sua tarefa é avaliar a relevância de um DOCUMENTO em relação a uma CONSULTA (QUERY), atribuindo um score de relevância em {0, 1, 2}, conforme as regras abaixo.

### REGRAS PARA O SCORE ###:
2 – RELEVANTE: O documento responde diretamente à consulta e contém a informação central solicitada. O conteúdo é suficiente para permitir ao usuário atingir o objetivo da busca. O documento não será considerado relevante para a consulta se apenas mencionar termos semelhantes, mas não tratar efetivamente do objeto da consulta.

1 – POUCO RELEVANTE: O documento menciona o tema da consulta ou trata de assunto relacionado, mas de forma parcial ou insuficiente para atender plenamente à necessidade informacional.

0 – IRRELEVANTE: O documento não trata do assunto da consulta ou aborda tema distinto/incompatível.
### ###

A sua resposta deverá ser apenas um JSON válido, sem qualquer texto antes ou depois, e deve conter apenas os atributos "justificativa" e "score", nessa ordem.

### REGRAS PARA O JSON ###
"justificativa": Texto explicando se o documento é IRRELEVANTE, POUCO RELEVANTE ou RELEVANTE para a consulta.

"score": O valor do score (0, 1 ou 2) para o grau de relevância estimado.
### ###
""".strip()

In [8]:
MSG_USUARIO = """
Avalie a relevância do documento para a query.

QUERY: {QUERY}

DOCUMENTO:
{DOCUMENTO}
""".strip()

In [9]:
PROMPTS_SISTEMA_PARA_TESTAR = {
    "simple": MSG_SISTEMA_SIMPLE_PROMPT,
    "cot": MSG_SISTEMA_COT_PROMPT
}

# 2. Pares únicos de QUERY_KEY/DOC_KEY para avaliar

In [19]:
# 1) O qrels já tem apenas pares únicos. Basta carregá-lo e filtrar as colunas QUERY_KEY e DOC_KEY
df_qrels = pd.read_csv(ARQUIVO_QREL)

pairs = df_qrels[["QUERY_KEY", "DOC_KEY"]]
pairs = list(map(tuple, pairs[["QUERY_KEY", "DOC_KEY"]].to_numpy()))

print(f'Total de pares: {len(pairs)}')

Total de pares: 812


## 3. Funções para obter um texto a partir da QUERY_KEY e DOC_KEY

In [20]:
df_queries = pd.read_csv(ARQUIVO_QUERY)
df_docs = pd.read_csv(ARQUIVO_DOCS)

In [21]:
from formatador import html_to_plain_text

# criar dicionários para acesso rápido por chave
query_text_by_key = df_queries.set_index("KEY")["TEXT"].to_dict()
def query_key_to_query(key):
    return query_text_by_key[key]

doc_html_by_key = df_docs.set_index("KEY")["TEXTONORMA"].to_dict()
def doc_key_to_doc(key):
    return html_to_plain_text(doc_html_by_key[key])

Testa o código para ver se é possível recuperar todas as queries e docs

In [22]:
for qkey, dkey in pairs:
    query_text = query_key_to_query(qkey)
    doc_text = doc_key_to_doc(dkey)

In [23]:
print(doc_text)

Constitui grupo de trabalho com objetivo de elaborar Manual de Auditoria Financeira. 
 
             O SECRETÁRIO-GERAL DE CONTROLE EXTERNO DO TRIBUNAL DE CONTAS DA UNIÃO, no uso de suas competências regulamentares e considerando o disposto no art. 33, inciso II, da Resolução-TCU n.º 253, de 21 de dezembro de 2012, resolve:
             Art. 1º Fica constituído o Grupo de Trabalho para a elaboração de Manual de Auditoria Financeira.
             Art. 2º O Manual de Auditoria Financeira deverá estar alinhado com as normas constitucionais e legais vigentes no Brasil, as Normas de Auditoria do TCU (NATs) e os Padrões de Auditoria Financeira, emitidos pela Organização Internacional de Entidades de Fiscalização Superior (Intosai).
             Art. 3º O Grupo de Trabalho deverá levar em consideração também as conclusões e recomendações decorrentes das consultorias realizadas no âmbito do Projeto de Fortalecimento da Auditoria Financeira, financiado pelo Acordo de Doação TF 099104 com o Banc

## 4. Anotação usando LLM

### 4.1. Carrega/cria o arquivo onde ficarão as avaliações

In [24]:
# Verifica se o arquivo existe
if os.path.exists(ARQUIVO_LLM_EVALS):
    df_llm_evals = pd.read_csv(ARQUIVO_LLM_EVALS)
else:
    # Cria o DataFrame com a estrutura desejada
    df_llm_evals = pd.DataFrame(columns=[
        'QUERY_KEY', 'DOC_KEY', 'AVALIACAO', 'MODELO', 'PROMPT', 'PAYLOAD_LLM'
    ])
    
    # Salva o novo DataFrame no arquivo
    df_llm_evals.to_csv(ARQUIVO_LLM_EVALS, index=False)

### 4.2. Função genérica para avaliar um par de query-documento usando algum modelo/prompt

In [25]:
def extract_token_limits_from_exception(llm_model, exc: Exception):
    error_text = str(exc)

    if llm_model == 'deepseek-chat':
        pattern = r'maximum context length is (\d+) tokens.*requested (\d+) tokens'
        match = re.search(pattern, error_text)

        if not match:
            return None

        max_context = int(match.group(1))
        requested = int(match.group(2))

        return max_context, requested

    if llm_model == 'sabiazinho-4-2026-01-06':
        pattern = r'too long:\s*(\d+)\s*tokens.*maximum supported is\s*(\d+)\s*tokens'
        match = re.search(pattern, error_text, re.IGNORECASE)

        if not match:
            return None

        requested = int(match.group(1))
        max_context = int(match.group(2))

        return max_context, requested

    if llm_model == 'gpt-5-mini-2025-08-07':
        pattern = r'limit of (\d+) tokens.*resulted in (\d+) tokens'
        match = re.search(pattern, error_text, re.IGNORECASE)

        if not match:
            return None

        max_context = int(match.group(1))
        requested = int(match.group(2))

        return max_context, requested

In [39]:
def avaliar(query_key, doc_key, llm_model, msg_sistema, client, perc_max_texto=1.0):
    query_text = query_key_to_query(query_key)
    doc_text = doc_key_to_doc(doc_key)

    idx_truncamento = int(len(doc_text) * perc_max_texto)
    doc_text_truncado = doc_text[:idx_truncamento]
    
    try:
        response = client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "system", "content": msg_sistema},
                {"role": "user", "content": MSG_USUARIO.format(QUERY=query_text, DOCUMENTO=doc_text_truncado)}
            ],
            #temperature=0, # não funciona com o gpt-5-mini
            stream=False
        )
        content = response.choices[0].message.content
    except Exception as e:
        result = extract_token_limits_from_exception(llm_model, e)
        
        if result is None:
            print(e)
        else:
            contexto_max, contexto_pedido = result
            percentual_truncar = 0.95 * perc_max_texto * contexto_max/contexto_pedido
            print(f'{query_key}/{doc_key}. Solicitado {contexto_pedido} tokens, mas modelo suporta {contexto_max}. Tenta novamente truncando com fator {percentual_truncar}')
            return avaliar(query_key, doc_key, llm_model, msg_sistema, client, percentual_truncar)

    # Pode ser que o LLM responda com algo diferente do json. Nesse caso, seta o score para -1 e salva para reavaliarmos novamente depois
    try:
        # Sanitiza content. As vezes retorna iniciando e terminando com sinalizadores de json
        content = content.replace('```json', '').replace('```', '')
        score = json.loads(content)['score']
    except Exception as e:
        score = -1

    return score, content

### 4.4. Loop para avaliação em todos os modelos/prompts/pares de documentos-query

In [27]:
for qkey, dkey in pairs:
    for nome_prompt, prompt_sistema in PROMPTS_SISTEMA_PARA_TESTAR.items():
        for config_llm in CONFIGURACOES_LLM_PARA_TESTAR:
            model = config_llm['model']
            client = config_llm['client']
        
            # Verifica se já existe essa combinação no df_llm_evals
            ja_foi_avaliado = (
                (df_llm_evals['QUERY_KEY'] == qkey) &
                (df_llm_evals['DOC_KEY'] == dkey) &
                (df_llm_evals['MODELO'] == model) &
                (df_llm_evals['PROMPT'] == nome_prompt)
            )

            if ja_foi_avaliado.any():
                # Esse modelo já tem avaliação, passa para o próximo
                score_do_modelo = df_llm_evals.loc[ja_foi_avaliado, 'AVALIACAO'].values[0]
                print(f'Já possui avaliação para {qkey}/{dkey}/{model}/{nome_prompt}: {score_do_modelo}')
                continue

            # Se não foi avaliado, avalia e extrai score e payload
            score, payload = avaliar(qkey, dkey, model, prompt_sistema, client)
            print(f'Avaliando para {qkey}/{dkey}/{model}/{nome_prompt}: {score}')

            # Cria a nova avaliação
            nova_avaliacao = {
                'QUERY_KEY': qkey,
                'DOC_KEY': dkey,
                'AVALIACAO': score,
                'MODELO': model,
                'PROMPT': nome_prompt,
                'PAYLOAD_LLM': payload
            }

            # Adiciona ao DataFrame
            df_llm_evals = pd.concat([df_llm_evals, pd.DataFrame([nova_avaliacao])], ignore_index=True)
    
            # Salva no arquivo
            df_llm_evals.to_csv(ARQUIVO_LLM_EVALS, index=False)

Já possui avaliação para 1/NORMA-21217/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-21217/sabiazinho-4-2026-01-06/cot: 2
Já possui avaliação para 1/NORMA-19574/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-19574/sabiazinho-4-2026-01-06/cot: 2
Já possui avaliação para 1/NORMA-21779/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-21779/sabiazinho-4-2026-01-06/cot: 2
Já possui avaliação para 1/NORMA-21754/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-21754/sabiazinho-4-2026-01-06/cot: 2
Já possui avaliação para 1/NORMA-21621/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-21621/sabiazinho-4-2026-01-06/cot: 2
Já possui avaliação para 1/NORMA-21432/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-21432/sabiazinho-4-2026-01-06/cot: 2
Já possui avaliação para 1/NORMA-23521/sabiazinho-4-2026-01-06/simple: 2
Já possui avaliação para 1/NORMA-23521/sabiazinho-4-2026-01-06/cot: 2

In [35]:
score, payload = avaliar(5, 'NORMA-25902', 'sabiazinho-4-2026-01-06', MSG_SISTEMA_SIMPLE_PROMPT, client_sabia)

[{'role': 'system', 'content': 'Você é um avaliador de relevância de normas do Tribunal de Contas da União (TCU).\n\nSua tarefa é avaliar a relevância de um DOCUMENTO em relação a uma CONSULTA (QUERY), atribuindo um score de relevância em {0, 1, 2}, conforme as regras abaixo.\n\n### REGRAS PARA O SCORE ###:\n2 – RELEVANTE: O documento responde diretamente à consulta e contém a informação central solicitada. O conteúdo é suficiente para permitir ao usuário atingir o objetivo da busca. O documento não será considerado relevante para a consulta se apenas mencionar termos semelhantes, mas não tratar efetivamente do objeto da consulta.\n\n1 – POUCO RELEVANTE: O documento menciona o tema da consulta ou trata de assunto relacionado, mas de forma parcial ou insuficiente para atender plenamente à necessidade informacional.\n\n0 – IRRELEVANTE: O documento não trata do assunto da consulta ou aborda tema distinto/incompatível.\n### ###\n\nA sua resposta deverá ser apenas um JSON válido, sem qualqu

In [38]:
payload

'{"score":0}'